# Guided Design Tutorial Notebook

Notebook companion to the [Guided Design Tutorial](../developers-docs/guided-design-tutorial.md) and the focused [Programmatic Zomic Visualization](../developers-docs/programmatic-zomic-visualization.md) reference.

Use the Markdown tutorial when you want the shortest checked operator story. Use this notebook when you want:

- the richest runnable companion for the same checked Sc-Zn walkthrough
- inline artifact inspection and execution controls
- a documented preview-vs-refresh path for the repo-owned visualization helper
- deeper branch guidance for the shipped LLM surfaces without leaving the walkthrough

This notebook keeps one `Sc-Zn deterministic spine`:
`.zomic -> raw export -> orbit library -> candidates`.

The additive branches stay attached to that same authority chain:

```text
Sc-Zn deterministic spine
  .zomic -> export-zomic -> preview-zomic -> generate -> screen -> hifi-validate -> hifi-rank -> report
    |
    +-- Same-System Companion Lane
    |     llm-generate -> llm-evaluate
    |
    `-- Translation and external benchmark branch
          llm-translate -> llm-translated-benchmark-freeze -> llm-register-external-target -> llm-external-benchmark
```


## Environment Assumptions

- Launch the notebook from the repo root or from the `materials-discovery/` workspace.
- Run `uv sync --extra dev` before expecting command cells to execute successfully.
- The checked Sc-Zn deterministic path needs a local Java runtime for `mdisc export-zomic` and the Zomic-backed `mdisc generate` path.
- Real/native providers remain optional. The deterministic tutorial path uses checked repo artifacts even if you keep execution preview-only.
- The notebook defaults to preview-safe mode: commands print exactly what they would run, but they do not mutate artifacts until you enable the relevant `RUN_*` flag.
- External-target and external-benchmark cells stay intentionally conservative because the shipped example specs still contain placeholder snapshot/revision values that you must replace locally.


### Helper conventions for the first code cell

The long setup cell right after this section is the notebook's control panel. It does four things before any workflow-specific stage runs:

1. resolves `WORKDIR` so the notebook works from either the repo root or the `materials-discovery/` subtree
2. adds `materials-discovery/src` to `sys.path` and imports the programmatic preview helpers
3. defines the small utility functions (`run`, `read_json`, `read_jsonl`, `maybe_read_json`, `path_status`) that every later cell uses
4. defines `preview_checked_design()` so the notebook can switch between checked-preview mode and refresh-preview mode with one flag

Two conventions are worth keeping in mind:

- `run(...)` always prints the exact CLI command, even when execution is disabled
- the `maybe_*` helpers return `{exists, path, ...}` wrappers so explanatory cells can stay readable when optional artifacts have not been created yet


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import shlex
import subprocess
import sys
import tempfile

try:
    from IPython.display import HTML, display
except Exception:  # pragma: no cover - notebook fallback
    HTML = None
    def display(*_args, **_kwargs):
        return None

ROOT = Path.cwd().resolve()
WORKDIR = ROOT / "materials-discovery" if (ROOT / "materials-discovery").exists() else ROOT
assert (WORKDIR / "configs").exists(), f"Could not find materials-discovery workspace from {ROOT}"

SRC = WORKDIR / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from materials_discovery.visualization import preview_raw_export, preview_zomic_design

RUN_PIPELINE = False
RUN_SAME_SYSTEM_LLM = False
RUN_TRANSLATED_BENCHMARK = False
RUN_EXTERNAL_TARGETS = False
RUN_EXTERNAL_BENCHMARK = False
REFRESH_PREVIEW = False
EMBED_PREVIEW = True
SHOW_PREVIEW_LABELS = True
TMPDIR = Path(tempfile.gettempdir())


def run(cmd: list[str], *, execute: bool | None = None, cwd: Path = WORKDIR):
    if execute is None:
        execute = RUN_PIPELINE
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    if not execute:
        print("(preview only; enable the relevant RUN_* flag to execute this cell)")
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, check=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result


def rel(path: str | Path) -> str:
    path = Path(path).resolve()
    try:
        return str(path.relative_to(WORKDIR))
    except ValueError:
        return str(path)


def read_json(path: str | Path):
    resolved = WORKDIR / Path(path)
    return json.loads(resolved.read_text())


def maybe_read_json(path: str | Path):
    resolved = WORKDIR / Path(path)
    if not resolved.exists():
        return {"exists": False, "path": rel(resolved)}
    payload = json.loads(resolved.read_text())
    return {
        "exists": True,
        "path": rel(resolved),
        "payload": payload,
    }


def read_jsonl(path: str | Path, limit: int | None = None):
    resolved = WORKDIR / Path(path)
    rows = [json.loads(line) for line in resolved.read_text().splitlines() if line.strip()]
    return rows if limit is None else rows[:limit]


def maybe_read_jsonl(path: str | Path, limit: int | None = None):
    resolved = WORKDIR / Path(path)
    if not resolved.exists():
        return {"exists": False, "path": rel(resolved), "rows": []}
    return {
        "exists": True,
        "path": rel(resolved),
        "rows": read_jsonl(resolved.relative_to(WORKDIR), limit=limit),
    }


def path_status(paths: dict[str, str | Path]):
    status = {}
    for name, raw_path in paths.items():
        resolved = (WORKDIR / Path(raw_path)).resolve()
        status[name] = {
            "path": rel(resolved),
            "exists": resolved.exists(),
        }
    return status


def preview_checked_design(*, refresh: bool | None = None, show_labels: bool | None = None):
    refresh = REFRESH_PREVIEW if refresh is None else refresh
    show_labels = SHOW_PREVIEW_LABELS if show_labels is None else show_labels
    out_path = TMPDIR / "sc_zn_tsai_bridge.notebook.viewer.html"
    if refresh:
        summary = preview_zomic_design(
            WORKDIR / "designs/zomic/sc_zn_tsai_bridge.yaml",
            out_path=out_path,
            show_labels=show_labels,
        )
    else:
        summary = preview_raw_export(
            WORKDIR / "data/prototypes/generated/sc_zn_tsai_bridge.raw.json",
            out_path=out_path,
            show_labels=show_labels,
        )
    payload = summary.model_dump(mode="json")
    if EMBED_PREVIEW and HTML is not None:
        display(HTML(Path(payload["html_path"]).read_text()))
    return payload


WORKDIR


PosixPath('/Users/nikolaosvasiloglou/github-repos/vzome/materials-discovery')

## 1. Workspace Setup

Run the standard developer sync first. Leave the `RUN_*` flags `False` if you only want to preview commands and inspect the checked repo artifacts.

The setup cell below is intentionally simple:

- it previews `uv sync --extra dev` through the same `run(...)` helper used everywhere else in the notebook
- it prints the current flag values so you can confirm which branches are armed before anything expensive runs

Use the flags like this:

- `RUN_PIPELINE = True` if you want to rerun the deterministic export/generate/screen/validate/rank/report steps
- `REFRESH_PREVIEW = True` if you want the notebook preview to regenerate the export before rendering
- `RUN_SAME_SYSTEM_LLM = True` if you want to execute the Sc-Zn LLM lane
- `RUN_TRANSLATED_BENCHMARK = True` if you want to execute the translation/freeze inspect path rather than only inspect shipped demo artifacts
- `RUN_EXTERNAL_TARGETS = True` only after replacing the placeholder external-target specs with real local snapshots and pinned revisions
- `RUN_EXTERNAL_BENCHMARK = True` only after the translated benchmark set exists and the external target has passed its smoke check
- `EMBED_PREVIEW` controls whether the HTML viewer is rendered inline in the notebook
- `SHOW_PREVIEW_LABELS` toggles the labeled-point overlay in the preview helper


In [2]:
run(["uv", "sync", "--extra", "dev"])

{
    "RUN_PIPELINE": RUN_PIPELINE,
    "REFRESH_PREVIEW": REFRESH_PREVIEW,
    "RUN_SAME_SYSTEM_LLM": RUN_SAME_SYSTEM_LLM,
    "RUN_TRANSLATED_BENCHMARK": RUN_TRANSLATED_BENCHMARK,
    "RUN_EXTERNAL_TARGETS": RUN_EXTERNAL_TARGETS,
    "RUN_EXTERNAL_BENCHMARK": RUN_EXTERNAL_BENCHMARK,
}


$ uv sync --extra dev
(preview only; enable the relevant RUN_* flag to execute this cell)


{'RUN_PIPELINE': False,
 'REFRESH_PREVIEW': False,
 'RUN_SAME_SYSTEM_LLM': False,
 'RUN_TRANSLATED_BENCHMARK': False,
 'RUN_EXTERNAL_TARGETS': False,
 'RUN_EXTERNAL_BENCHMARK': False}

## About This Design

The Sc-Zn system in this tutorial is a Tsai-type icosahedral quasicrystal approximant — concentric polyhedral shells (inner icosahedron, pentagonal dodecahedron, outer icosidodecahedron) that tile 3D space with quasiperiodic order. The `.zomic` file encodes a hand-designed bridge motif that the pipeline then expands into machine-readable orbit-library candidates.

See [About This Design](../developers-docs/guided-design-tutorial.md#about-this-design) in the full tutorial for the Tsai cluster explanation, the IUCrJ 2016 Sc-Zn citation, and the bridge concept.

## 2. Know the Worked Example

The deterministic walkthrough uses these checked inputs:

- `configs/systems/sc_zn_zomic.yaml`
- `designs/zomic/sc_zn_tsai_bridge.yaml`
- `designs/zomic/sc_zn_tsai_bridge.zomic`
- `data/prototypes/sc_zn_tsai_sczn6.json`

Geometry authority stays on one chain:

`sc_zn_tsai_bridge.zomic` -> `sc_zn_tsai_bridge.raw.json` -> `sc_zn_tsai_bridge.json` -> candidate/report artifacts

The `.zomic` file is the editable source. Everything downstream is derived from it.

The path inventory cell below is mostly an orientation map. It groups:

- the deterministic Sc-Zn authority chain
- the docs pages that explain the same example in prose
- the same-system LLM configs for the additive Sc-Zn lane
- the Al-Cu-Fe translation/external-benchmark specs used later as shipped fixture-backed examples

If one of those paths surprises you, stop here and verify the workspace before moving on to execution.


In [3]:
paths = {
    "system_config": WORKDIR / "configs/systems/sc_zn_zomic.yaml",
    "design_yaml": WORKDIR / "designs/zomic/sc_zn_tsai_bridge.yaml",
    "design_source": WORKDIR / "designs/zomic/sc_zn_tsai_bridge.zomic",
    "anchor_prototype": WORKDIR / "data/prototypes/sc_zn_tsai_sczn6.json",
    "raw_export": WORKDIR / "data/prototypes/generated/sc_zn_tsai_bridge.raw.json",
    "orbit_library": WORKDIR / "data/prototypes/generated/sc_zn_tsai_bridge.json",
    "markdown_tutorial": WORKDIR / "developers-docs/guided-design-tutorial.md",
    "visualization_reference": WORKDIR / "developers-docs/programmatic-zomic-visualization.md",
    "same_system_mock": WORKDIR / "configs/systems/sc_zn_llm_mock.yaml",
    "same_system_local": WORKDIR / "configs/systems/sc_zn_llm_local.yaml",
    "translated_benchmark_spec": WORKDIR / "configs/llm/al_cu_fe_translated_benchmark_freeze.yaml",
    "external_target_spec": WORKDIR / "configs/llm/al_cu_fe_external_cif_target.yaml",
    "external_benchmark_spec": WORKDIR / "configs/llm/al_cu_fe_external_benchmark.yaml",
}

{name: rel(path) for name, path in paths.items()}


{'system_config': 'configs/systems/sc_zn_zomic.yaml',
 'design_yaml': 'designs/zomic/sc_zn_tsai_bridge.yaml',
 'design_source': 'designs/zomic/sc_zn_tsai_bridge.zomic',
 'anchor_prototype': 'data/prototypes/sc_zn_tsai_sczn6.json',
 'raw_export': 'data/prototypes/generated/sc_zn_tsai_bridge.raw.json',
 'orbit_library': 'data/prototypes/generated/sc_zn_tsai_bridge.json',
 'markdown_tutorial': 'developers-docs/guided-design-tutorial.md',
 'visualization_reference': 'developers-docs/programmatic-zomic-visualization.md',
 'same_system_mock': 'configs/systems/sc_zn_llm_mock.yaml',
 'same_system_local': 'configs/systems/sc_zn_llm_local.yaml',
 'translated_benchmark_spec': 'configs/llm/al_cu_fe_translated_benchmark_freeze.yaml',
 'external_target_spec': 'configs/llm/al_cu_fe_external_cif_target.yaml',
 'external_benchmark_spec': 'configs/llm/al_cu_fe_external_benchmark.yaml'}

### Reading the Zomic Source

The `.zomic` file uses three orbit labels that appear throughout the pipeline:

| Label | Physical part | Species |
|-------|---------------|----------|
| `pent` | Pentagonal ring | Sc, Zn |
| `frustum` | Frustum connectors | Zn, Sc |
| `joint` | Outer joint sites | Zn |

See [Section 2.1](../developers-docs/guided-design-tutorial.md#21-reading-the-zomic-design-source) in the full tutorial for the annotated Zomic block walkthrough.

## 3. Export the Zomic Design

This compiles the Zomic source into the raw labeled geometry export and the orbit-library JSON that `mdisc generate` consumes.

The two outputs matter for different reasons:

- the raw export is the preview and geometry-inspection artifact used by the notebook viewer
- the orbit-library JSON is the grouped orbit representation that feeds candidate generation

The summary dictionary below intentionally pulls a few fields from each file so you can verify both layers before you move on.


In [4]:
run([
    "uv", "run", "mdisc", "export-zomic",
    "--design", "designs/zomic/sc_zn_tsai_bridge.yaml",
])

raw = read_json("data/prototypes/generated/sc_zn_tsai_bridge.raw.json")
orbit = read_json("data/prototypes/generated/sc_zn_tsai_bridge.json")

{
    "raw_zomic_file": raw.get("zomic_file"),
    "raw_labeled_points": len(raw.get("labeled_points", [])),
    "raw_segments": len(raw.get("segments", [])),
    "orbit_source_kind": orbit.get("source_kind"),
    "orbit_count": len(orbit.get("orbits", [])),
    "anchor_orbit_summary": orbit.get("anchor_orbit_summary"),
}


$ uv run mdisc export-zomic --design designs/zomic/sc_zn_tsai_bridge.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)


{'raw_zomic_file': '/Users/nikolaosvasiloglou/github-repos/vzome/materials-discovery/designs/zomic/sc_zn_tsai_bridge.zomic',
 'raw_labeled_points': 52,
 'raw_segments': 52,
 'orbit_source_kind': 'zomic_export_anchor_expanded',
 'orbit_count': 5,
 'anchor_orbit_summary': {'min_votes': 2,
  'selected_orbits': ['tsai_zn7',
   'tsai_sc1',
   'tsai_zn6',
   'tsai_zn5',
   'tsai_zn4'],
  'selected_site_count': 100,
  'site_target': 100,
  'strategy': 'seed_orbit_expand'}}

## 4. Programmatic Preview of the Checked Geometry

This is the notebook's preview-vs-refresh path:

- `preview_raw_export(...)` renders the checked raw export exactly as it exists now
- `preview_zomic_design(...)` refreshes through `export-zomic` first, then renders the updated export

Keep `REFRESH_PREVIEW = False` when you only want the normal checked inspection path. Flip it to `True` when you intentionally want the notebook to regenerate the export before rendering.


In [5]:
preview_summary = preview_checked_design()
preview_summary


{'design_path': None,
 'raw_export_path': '/Users/nikolaosvasiloglou/github-repos/vzome/materials-discovery/data/prototypes/generated/sc_zn_tsai_bridge.raw.json',
 'html_path': '/private/var/folders/nb/7y9vtgkn05gblnsxg87f6s9h0000gn/T/sc_zn_tsai_bridge.notebook.viewer.html',
 'labeled_point_count': 52,
 'segment_count': 52,
 'opened_browser': False,
 'show_labels': True}

### Interactive 3D Orbit Visualization

The cells below render the same 100 anchor-library sites as interactive plotly
figures.  Hover over any site to see its label, anchor orbit, species, and
shell name.  If plotly is not installed the HTML canvas viewer above is the
fallback; install the `[viz]` extra to enable these figures:

```
uv pip install "materials-discovery[viz]"
```


In [ ]:
# 4.1 — Interactive orbit-colored scatter (VIZ-01)
try:
    from materials_discovery.visualization.plotly_3d import (
        orbit_figure,
        shell_figure,
        load_orbit_library,
    )
    _PLOTLY_AVAILABLE = True
except ImportError:
    _PLOTLY_AVAILABLE = False

if _PLOTLY_AVAILABLE:
    orbit_lib_data = load_orbit_library(
        WORKDIR / "data/prototypes/generated/sc_zn_tsai_bridge.json"
    )
    fig_orbit = orbit_figure(orbit_lib_data)
    fig_orbit.show(renderer="notebook_connected")
else:
    print("Install plotly for interactive 3D figures:")
    print('  uv pip install "materials-discovery[viz]"')
    print("Falling back to HTML canvas viewer above.")


In [ ]:
# 4.2 — Shell-decomposed Tsai cluster figure (VIZ-02)
if _PLOTLY_AVAILABLE:
    fig_shell = shell_figure(orbit_lib_data)
    fig_shell.show(renderer="notebook_connected")


## 5. Generate and Screen Candidates

The next two stages create the candidate population and apply the fast screening filters. The checked snapshot teaches the current artifact shape even if you do not rerun the commands.

The cell below samples one generated row and one screened row on purpose. That is enough to answer:

- what a generated candidate record looks like
- which manifest records generation-stage lineage
- where fast-screen calibration counts live
- which shortlist proxy fields survive into the screened output

If you want the whole batch later, inspect the `.jsonl` files directly instead of assuming the first row is a winner.


In [6]:
run([
    "uv", "run", "mdisc", "generate",
    "--config", "configs/systems/sc_zn_zomic.yaml",
    "--count", "32",
])

candidate = read_jsonl("data/candidates/sc_zn_candidates.jsonl", limit=1)[0]
generate_manifest = read_json("data/manifests/sc_zn_generate_manifest.json")

{
    "candidate_id": candidate.get("candidate_id"),
    "composition": candidate.get("composition"),
    "prototype_key": candidate.get("provenance", {}).get("prototype_key"),
    "prototype_source_kind": candidate.get("provenance", {}).get("prototype_source_kind"),
    "site_count": len(candidate.get("sites", [])),
    "generate_stage": generate_manifest.get("stage"),
}

run([
    "uv", "run", "mdisc", "screen",
    "--config", "configs/systems/sc_zn_zomic.yaml",
])

screen_cal = read_json("data/calibration/sc_zn_screen_calibration.json")
screened = read_jsonl("data/screened/sc_zn_screened.jsonl", limit=1)[0]

{
    "input_count": screen_cal.get("input_count"),
    "passed_count": screen_cal.get("passed_count"),
    "shortlisted_count": screen_cal.get("shortlisted_count"),
    "first_shortlist_rank": screened.get("screen", {}).get("shortlist_rank"),
    "first_energy_proxy_ev_per_atom": screened.get("screen", {}).get("energy_proxy_ev_per_atom"),
    "first_min_distance_proxy": screened.get("screen", {}).get("min_distance_proxy"),
}


$ uv run mdisc generate --config configs/systems/sc_zn_zomic.yaml --count 32
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc screen --config configs/systems/sc_zn_zomic.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)


{'input_count': 30,
 'passed_count': 20,
 'shortlisted_count': 4,
 'first_shortlist_rank': 1,
 'first_energy_proxy_ev_per_atom': -2.778674,
 'first_min_distance_proxy': 0.751937}

### What the screening numbers mean

`energy_proxy_ev_per_atom` estimates cohesive energy per atom — more negative means more stable. `min_distance_proxy` catches unphysically close atoms. Of 30 input candidates, 20 pass both thresholds and 4 enter the shortlist for expensive validation.

See [Section 5](../developers-docs/guided-design-tutorial.md#5-screen-the-candidate-population) in the full tutorial for the metric-by-metric explanation.

## 6. High-Fidelity Validation, Ranking, and Report

These stages tell you whether the batch is actually promising or merely interesting. The current checked Sc-Zn snapshot is useful precisely because it ends in `hold`/`watch` rather than a fake success story.

The notebook reads one validated row plus the batch report because they answer different questions:

- the validated row carries candidate-level digital-validation signals and warnings
- `report.json` carries the batch-level release gate, counts, and recommendation line

Keeping those two artifacts separate makes it easier to see why a batch can contain interesting candidates without crossing the release threshold.


In [7]:
run([
    "uv", "run", "mdisc", "hifi-validate",
    "--config", "configs/systems/sc_zn_zomic.yaml",
    "--batch", "all",
])
run([
    "uv", "run", "mdisc", "hifi-rank",
    "--config", "configs/systems/sc_zn_zomic.yaml",
])
run([
    "uv", "run", "mdisc", "report",
    "--config", "configs/systems/sc_zn_zomic.yaml",
])

validated = read_jsonl("data/hifi_validated/sc_zn_all_validated.jsonl", limit=1)[0]
report = read_json("data/reports/sc_zn_report.json")

{
    "validated_candidate_id": validated.get("candidate_id"),
    "geometry_prefilter_pass": validated.get("digital_validation", {}).get("geometry_prefilter_pass"),
    "phonon_imaginary_modes": validated.get("digital_validation", {}).get("phonon_imaginary_modes"),
    "md_stability_score": validated.get("digital_validation", {}).get("md_stability_score"),
    "xrd_confidence": validated.get("digital_validation", {}).get("xrd_confidence"),
    "ranked_count": report.get("ranked_count"),
    "reported_count": report.get("reported_count"),
    "release_gate": report.get("release_gate"),
    "top_recommendation": (report.get("summary") or {}).get("recommendation"),
}


$ uv run mdisc hifi-validate --config configs/systems/sc_zn_zomic.yaml --batch all
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc hifi-rank --config configs/systems/sc_zn_zomic.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc report --config configs/systems/sc_zn_zomic.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)


{'validated_candidate_id': 'md_000006',
 'geometry_prefilter_pass': False,
 'phonon_imaginary_modes': 99,
 'md_stability_score': 0.0,
 'xrd_confidence': 0.0,
 'ranked_count': 4,
 'reported_count': 4,
 'release_gate': {'enough_synthesis_candidates': False,
  'ready_for_experimental_pack': False,
  'top_distinctiveness_gate': False,
  'top_ood_gate': False,
  'top_stability_gate': False,
  'top_xrd_confidence_gate': False},
 'top_recommendation': None}

### What the validation gates mean

All gates are False in this batch — that is the expected result for a mock-backend early-stage run. `geometry_prefilter_pass` catches crowding, `phonon_imaginary_modes` checks dynamical stability, `md_stability_score` measures structural drift, and `xrd_confidence` compares diffraction patterns. The pipeline is showing you the unfiltered signal, not a failure.

See [Section 6](../developers-docs/guided-design-tutorial.md#6-run-high-fidelity-validation) in the full tutorial for the gate checklist and honest-batch framing.

## 7. How to Read the Current Evidence

The checked artifact snapshot is dated, so treat exact counts as a point-in-time reading, not a timeless promise.

What matters operationally:

- `shortlisted_count` tells you how many candidates survived fast screening strongly enough to justify expensive validation.
- `geometry_prefilter_pass` is the first cheap crowding gate before the expensive phonon/MD path.
- `phonon_imaginary_modes` and `md_stability_score` are stability warnings, not decorative metrics.
- `recommendation`, `priority`, `risk_flags`, `stability_probability`, and `release_gate` are the operator-facing summary of whether the run is ready for serious follow-up.

For the current checked Sc-Zn run, the honest reading is still: the batch is useful, but it remains a watchlist rather than a promotion-ready set.


## 8. Same-System Companion Lane

Use this branch when you want to stay inside the Sc-Zn family while adding an LLM proposal or assessment lane on top of the checked Zomic-backed workflow.

The same authority rule still applies: `.zomic` remains the geometry source, the deterministic export/generate/screen/validate/rank/report path remains the cleanest first read, and the LLM lane is additive around that evidence rather than a replacement for it.

The next cell uses two configs on purpose:

- `sc_zn_llm_mock.yaml` keeps proposal generation tutorial-safe with fixture outputs and a mock backend
- `sc_zn_llm_local.yaml` documents the real local/OpenAI-compatible serving lane at `http://localhost:8000`, including the `specialized_materials` evaluation lane

When `RUN_SAME_SYSTEM_LLM = False`, the artifact-status and preview dictionaries still teach the contract by showing where candidates, manifests, run records, and evaluation outputs would land.


In [8]:
run([
    "uv", "run", "mdisc", "llm-generate",
    "--config", "configs/systems/sc_zn_llm_mock.yaml",
    "--count", "5",
    "--out", "data/candidates/sc_zn_llm_candidates.jsonl",
], execute=RUN_SAME_SYSTEM_LLM)

run([
    "uv", "run", "mdisc", "llm-evaluate",
    "--config", "configs/systems/sc_zn_llm_local.yaml",
    "--batch", "all",
], execute=RUN_SAME_SYSTEM_LLM)

same_system_artifacts = path_status({
    "llm_candidate_output": "data/candidates/sc_zn_llm_candidates.jsonl",
    "llm_generate_manifest": "data/manifests/sc_zn_llm_generate_manifest.json",
    "llm_runs_root": "data/llm_runs",
    "llm_evaluated_output": "data/llm_evaluated/sc_zn_all_llm_evaluated.jsonl",
    "llm_evaluations_root": "data/llm_evaluations",
})

{
    "configs": [
        rel("configs/systems/sc_zn_llm_mock.yaml"),
        rel("configs/systems/sc_zn_llm_local.yaml"),
    ],
    "artifact_status": same_system_artifacts,
    "candidate_preview": maybe_read_jsonl("data/candidates/sc_zn_llm_candidates.jsonl", limit=1),
    "llm_evaluated_preview": maybe_read_jsonl("data/llm_evaluated/sc_zn_all_llm_evaluated.jsonl", limit=1),
}


$ uv run mdisc llm-generate --config configs/systems/sc_zn_llm_mock.yaml --count 5 --out data/candidates/sc_zn_llm_candidates.jsonl
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc llm-evaluate --config configs/systems/sc_zn_llm_local.yaml --batch all
(preview only; enable the relevant RUN_* flag to execute this cell)


{'configs': ['/Users/nikolaosvasiloglou/github-repos/vzome/configs/systems/sc_zn_llm_mock.yaml',
  '/Users/nikolaosvasiloglou/github-repos/vzome/configs/systems/sc_zn_llm_local.yaml'],
 'artifact_status': {'llm_candidate_output': {'path': 'data/candidates/sc_zn_llm_candidates.jsonl',
   'exists': False},
  'llm_generate_manifest': {'path': 'data/manifests/sc_zn_llm_generate_manifest.json',
   'exists': False},
  'llm_runs_root': {'path': 'data/llm_runs', 'exists': False},
  'llm_evaluated_output': {'path': 'data/llm_evaluated/sc_zn_all_llm_evaluated.jsonl',
   'exists': False},
  'llm_evaluations_root': {'path': 'data/llm_evaluations', 'exists': False}},
 'candidate_preview': {'exists': False,
  'path': 'data/candidates/sc_zn_llm_candidates.jsonl',
  'rows': []},
 'llm_evaluated_preview': {'exists': False,
  'path': 'data/llm_evaluated/sc_zn_all_llm_evaluated.jsonl',
  'rows': []}}

### What the LLM companion lane does

`llm-generate` uses a language model as an alternate proposal source — it replaces only the proposal step, not the validation evidence chain. `llm-evaluate` adds an assessment layer (synthesis feasibility, precursor availability) on top of ranked candidates without replacing deterministic validation.

See [Section 9.1](../developers-docs/guided-design-tutorial.md#91-same-system-companion-lane) in the full tutorial for the full explain-then-command-then-annotate walkthrough.

## 9. Why this branch switches to Al-Cu-Fe

The notebook keeps the deterministic spine on Sc-Zn because the checked design, raw export, and report evidence are anchored there.

The translation and external benchmark branch switches to Al-Cu-Fe on purpose because the repo already ships demo translation bundles, freeze specs, and external benchmark example specs for that family. Treat that as a fixture-backed context switch, not as a new geometry authority chain.


## 10. Translation and external benchmark branch

This branch moves across four artifact roots:

- `data/llm_translation_exports/`
- `data/benchmarks/llm_external_sets/`
- `data/llm_external_models/`
- `data/benchmarks/llm_external/`

Use the cells below to preview or inspect the shipped path in stages.

Read the branch as one chain rather than four disconnected command families:

1. export translated bundles from an internal ranked candidate set
2. freeze a stable translated benchmark pack
3. register and smoke a downloadable external runtime
4. compare that external runtime against the repo's internal control lane

Only the first two steps are intended to be inspectable out of the box from shipped demo artifacts. The last two depend on a real local snapshot and should stay preview-only until you replace the placeholder example specs.


### Step A — Inspect translation bundles

`llm-translate` exports one target family at a time from an internal ranked candidate set into `data/llm_translation_exports/{export_id}/`.
`llm-translate-inspect` is the quickest way to sanity-check the resulting `manifest.json` and `inventory.jsonl` before freezing anything into a benchmark pack.

The cell below points at the shipped Phase 34 Al-Cu-Fe demo CIF bundle and also checks the sibling material-string bundle so you can see how the notebook treats both target families as parallel exports from the same ranked source.


In [9]:
run([
    "uv", "run", "mdisc", "llm-translate",
    "--config", "configs/systems/al_cu_fe.yaml",
    "--input", "data/ranked/al_cu_fe_ranked.jsonl",
    "--target", "cif",
    "--export-id", "al_cu_fe_ranked_cif_v1",
], execute=RUN_TRANSLATED_BENCHMARK)

run([
    "uv", "run", "mdisc", "llm-translate-inspect",
    "--manifest", "data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/manifest.json",
], execute=RUN_TRANSLATED_BENCHMARK)

translation_bundle = {
    "demo_cif_manifest": maybe_read_json("data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/manifest.json"),
    "demo_cif_inventory": maybe_read_jsonl("data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/inventory.jsonl", limit=2),
    "bundle_root_status": path_status({
        "cif_bundle": "data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/manifest.json",
        "material_string_bundle": "data/llm_translation_exports/phase34_demo_al_cu_fe_material_string_v1/manifest.json",
    }),
}
translation_bundle


$ uv run mdisc llm-translate --config configs/systems/al_cu_fe.yaml --input data/ranked/al_cu_fe_ranked.jsonl --target cif --export-id al_cu_fe_ranked_cif_v1
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc llm-translate-inspect --manifest data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/manifest.json
(preview only; enable the relevant RUN_* flag to execute this cell)


{'demo_cif_manifest': {'exists': True,
  'path': 'data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/manifest.json',
  'payload': {'benchmark_context': None,
   'candidate_count': 1,
   'created_at_utc': '2026-04-07T06:13:04.688601Z',
   'export_id': 'phase34_demo_al_cu_fe_cif_v1',
   'exported_count': 1,
   'input_path': 'tests/fixtures/llm_translation/al_cu_fe_periodic_candidate.json',
   'inventory_path': 'data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/inventory.jsonl',
   'lossy_count': 0,
   'payload_dir': 'data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/payloads',
   'schema_version': 'llm-translation-bundle-manifest/v1',
   'source_lineage': None,
   'stage_manifest_path': None,
   'target_family': 'cif',
   'target_format': 'cif_text'}},
 'demo_cif_inventory': {'exists': True,
  'path': 'data/llm_translation_exports/phase34_demo_al_cu_fe_cif_v1/inventory.jsonl',
  'rows': [{'candidate_id': 'al_cu_fe_fixture_periodic_001',
    'composition': {'Al': 0.

### What the translation and benchmark branch does

`llm-translate` converts internal candidates to external formats (CIF, material strings). `llm-translated-benchmark-freeze` creates an immutable snapshot for benchmarking. `llm-register-external-target` registers a model for comparison. `llm-external-benchmark` runs the comparative scorecard.

See [Section 9.3](../developers-docs/guided-design-tutorial.md#93-translation-and-external-benchmark-branch) in the full tutorial for the step-by-step explanation.

### Step B — Freeze a translated benchmark set

`llm-translated-benchmark-freeze` turns one or more translation bundles into a stable benchmark slice under `data/benchmarks/llm_external_sets/{benchmark_set_id}/`.
The manifest tells you where the set came from; `included.jsonl` and `excluded.jsonl` tell you which translated rows survived the freeze rules and which ones were filtered out.

Treat this as the handoff between “I can export artifacts” and “I now have a benchmark contract that another runtime can be judged against.”


In [10]:
run([
    "uv", "run", "mdisc", "llm-translated-benchmark-freeze",
    "--spec", "configs/llm/al_cu_fe_translated_benchmark_freeze.yaml",
], execute=RUN_TRANSLATED_BENCHMARK)

run([
    "uv", "run", "mdisc", "llm-translated-benchmark-inspect",
    "--manifest", "data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json",
], execute=RUN_TRANSLATED_BENCHMARK)

translated_benchmark = {
    "freeze_spec": rel("configs/llm/al_cu_fe_translated_benchmark_freeze.yaml"),
    "benchmark_set_status": path_status({
        "manifest": "data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json",
        "included": "data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/included.jsonl",
        "excluded": "data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/excluded.jsonl",
    }),
    "manifest_preview": maybe_read_json("data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json"),
}
translated_benchmark


$ uv run mdisc llm-translated-benchmark-freeze --spec configs/llm/al_cu_fe_translated_benchmark_freeze.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc llm-translated-benchmark-inspect --manifest data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json
(preview only; enable the relevant RUN_* flag to execute this cell)


{'freeze_spec': '/Users/nikolaosvasiloglou/github-repos/vzome/configs/llm/al_cu_fe_translated_benchmark_freeze.yaml',
 'benchmark_set_status': {'manifest': {'path': 'data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json',
   'exists': False},
  'included': {'path': 'data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/included.jsonl',
   'exists': False},
  'excluded': {'path': 'data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/excluded.jsonl',
   'exists': False}},
 'manifest_preview': {'exists': False,
  'path': 'data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json'}}

### Step C — Register and smoke an external target

This is the first step that truly depends on a local external runtime. The shipped spec is an example template, not a promise that a downloadable model snapshot already exists on your machine.

The three main artifacts are:

- `registration.json` for immutable target identity and supported contracts
- `environment.json` for reproducibility and package/runtime capture
- `smoke_check.json` for “does this target answer the prompt/parser contract at all?”

Leave `RUN_EXTERNAL_TARGETS = False` until the spec points at a real `local_snapshot_path` and pinned revisions you control.


In [11]:
run([
    "uv", "run", "mdisc", "llm-register-external-target",
    "--spec", "configs/llm/al_cu_fe_external_cif_target.yaml",
], execute=RUN_EXTERNAL_TARGETS)

run([
    "uv", "run", "mdisc", "llm-inspect-external-target",
    "--model-id", "al_cu_fe_external_cif_demo",
], execute=RUN_EXTERNAL_TARGETS)

run([
    "uv", "run", "mdisc", "llm-smoke-external-target",
    "--model-id", "al_cu_fe_external_cif_demo",
], execute=RUN_EXTERNAL_TARGETS)

external_target_status = {
    "target_specs": [
        rel("configs/llm/al_cu_fe_external_cif_target.yaml"),
        rel("configs/llm/al_cu_fe_external_material_string_target.yaml"),
    ],
    "artifact_status": path_status({
        "registration": "data/llm_external_models/al_cu_fe_external_cif_demo/registration.json",
        "environment": "data/llm_external_models/al_cu_fe_external_cif_demo/environment.json",
        "smoke_check": "data/llm_external_models/al_cu_fe_external_cif_demo/smoke_check.json",
        "snapshot_manifest": "data/llm_external_snapshots/al_cu_fe_external_cif_demo/manifest.json",
    }),
    "note": "Replace the placeholder snapshot path and pinned revisions in the shipped example spec before enabling RUN_EXTERNAL_TARGETS.",
}
external_target_status


$ uv run mdisc llm-register-external-target --spec configs/llm/al_cu_fe_external_cif_target.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc llm-inspect-external-target --model-id al_cu_fe_external_cif_demo
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc llm-smoke-external-target --model-id al_cu_fe_external_cif_demo
(preview only; enable the relevant RUN_* flag to execute this cell)


{'target_specs': ['/Users/nikolaosvasiloglou/github-repos/vzome/configs/llm/al_cu_fe_external_cif_target.yaml',
  '/Users/nikolaosvasiloglou/github-repos/vzome/configs/llm/al_cu_fe_external_material_string_target.yaml'],
 'artifact_status': {'registration': {'path': 'data/llm_external_models/al_cu_fe_external_cif_demo/registration.json',
   'exists': False},
  'environment': {'path': 'data/llm_external_models/al_cu_fe_external_cif_demo/environment.json',
   'exists': False},
  'smoke_check': {'path': 'data/llm_external_models/al_cu_fe_external_cif_demo/smoke_check.json',
   'exists': False},
  'snapshot_manifest': {'path': 'data/llm_external_snapshots/al_cu_fe_external_cif_demo/manifest.json',
   'exists': False}},
 'note': 'Replace the placeholder snapshot path and pinned revisions in the shipped example spec before enabling RUN_EXTERNAL_TARGETS.'}

### Step D — Run the comparative external benchmark

`llm-external-benchmark` compares a registered external target against the translated benchmark set plus the internal control lane named in the spec. In the shipped example, that control is the promoted/default Al-Cu-Fe internal lane rather than a separate benchmark-only runtime.

The two files you usually inspect first are:

- `benchmark_summary.json` for the top-line recommendation and slice summaries
- `scorecard_by_case.jsonl` for per-case detail when the summary looks surprising

Enable `RUN_EXTERNAL_BENCHMARK` only after the translated benchmark set exists and the external target has passed its smoke check.


In [12]:
run([
    "uv", "run", "mdisc", "llm-external-benchmark",
    "--spec", "configs/llm/al_cu_fe_external_benchmark.yaml",
], execute=RUN_EXTERNAL_BENCHMARK)

run([
    "uv", "run", "mdisc", "llm-inspect-external-benchmark",
    "--summary", "data/benchmarks/llm_external/al_cu_fe_external_benchmark_v1/benchmark_summary.json",
], execute=RUN_EXTERNAL_BENCHMARK)

external_benchmark_status = {
    "benchmark_spec": rel("configs/llm/al_cu_fe_external_benchmark.yaml"),
    "artifact_status": path_status({
        "benchmark_set_manifest": "data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json",
        "benchmark_summary": "data/benchmarks/llm_external/al_cu_fe_external_benchmark_v1/benchmark_summary.json",
        "scorecard": "data/benchmarks/llm_external/al_cu_fe_external_benchmark_v1/scorecard_by_case.jsonl",
    }),
    "note": "Enable RUN_EXTERNAL_BENCHMARK only after the translated benchmark set exists and the external target has registered and passed its smoke check.",
}
external_benchmark_status


$ uv run mdisc llm-external-benchmark --spec configs/llm/al_cu_fe_external_benchmark.yaml
(preview only; enable the relevant RUN_* flag to execute this cell)
$ uv run mdisc llm-inspect-external-benchmark --summary data/benchmarks/llm_external/al_cu_fe_external_benchmark_v1/benchmark_summary.json
(preview only; enable the relevant RUN_* flag to execute this cell)


{'benchmark_spec': '/Users/nikolaosvasiloglou/github-repos/vzome/configs/llm/al_cu_fe_external_benchmark.yaml',
 'artifact_status': {'benchmark_set_manifest': {'path': 'data/benchmarks/llm_external_sets/al_cu_fe_translated_benchmark_v1/manifest.json',
   'exists': False},
  'benchmark_summary': {'path': 'data/benchmarks/llm_external/al_cu_fe_external_benchmark_v1/benchmark_summary.json',
   'exists': False},
  'scorecard': {'path': 'data/benchmarks/llm_external/al_cu_fe_external_benchmark_v1/scorecard_by_case.jsonl',
   'exists': False}},
 'note': 'Enable RUN_EXTERNAL_BENCHMARK only after the translated benchmark set exists and the external target has registered and passed its smoke check.'}

## 11. Follow-on Workflow Families

Keep these lighter workflow families explicit, but use the runbooks when you want the full operator surface.

The notebook intentionally stops at naming these families because each one has its own operational guardrails. Use the deterministic spine and the translation/external benchmark branch above first, then drop into the runbooks when you need a production operator workflow instead of a tutorial tour.

### Campaign governance

- `llm-suggest`
- `llm-approve`
- `llm-launch`
- `llm-replay`
- `llm-compare`

Use this when the deterministic and additive evidence has reached the point where you want an operator-governed proposal/approval/launch loop instead of another one-off tutorial branch.

### Serving and checkpoint operations

- `llm-serving-benchmark`
- `llm-register-checkpoint`
- `llm-list-checkpoints`
- `llm-promote-checkpoint`
- `llm-retire-checkpoint`

Use this when you need to compare serving lanes or manage adapted checkpoints without changing the base deterministic or translated benchmark contracts.

Reference docs:

- [Operator Runbook](../RUNBOOK.md)
- [LLM Integration](../developers-docs/llm-integration.md)


## 12. Preview, Execute, and Desktop vZome

Keep the three notebook modes straight:

| Need | Use this first | Why |
|------|----------------|-----|
| Inspect the checked geometry quickly | `preview_raw_export(...)` | Reads the current checked raw export and renders the normal repo-owned preview path. |
| Refresh the geometry before rendering | `preview_zomic_design(...)` with `REFRESH_PREVIEW = True` | Re-runs the export path intentionally before rendering. |
| Re-run deterministic CLI stages | `RUN_PIPELINE = True` | Executes the checked pipeline commands instead of only previewing them. |
| Author or deeply inspect the motif | desktop vZome | Desktop vZome remains the authoring and deeper-inspection tool. |

That split keeps the notebook honest: the repo preview is the default checked inspection path, the refresh path is deliberate, and desktop vZome still owns direct motif editing.

If a cell prints `(preview only; enable the relevant RUN_* flag...)`, the notebook is behaving correctly. Preview mode is there so you can study the shipped workflow without silently rewriting artifacts or assuming optional runtimes are available.

### Quick artifact glossary

When you are unsure where to start in an artifact directory, open the highest-level summary file first, `manifest.json` or `benchmark_summary.json` when present, and only then drill into the row-level `jsonl` files.

- `*.raw.json`: labeled geometry exported from a `.zomic` design for programmatic preview and downstream orbit extraction.
- orbit-library `.json`: grouped orbit or prototype library consumed by `mdisc generate`; this is the deterministic generation input after export.
- `*.jsonl`: line-delimited row artifacts where each line is one JSON object; use these when you want per-candidate or per-case detail without opening one giant file.
- `manifest.json`: stage or bundle lineage, configuration, and provenance metadata; read this before trusting derived artifacts.
- `inventory.jsonl`: translated-bundle inventory rows that trace each translated candidate payload and its source context.
- `included.jsonl` and `excluded.jsonl`: translated-benchmark freeze decisions showing which translated rows entered the benchmark set and which ones were left out.
- `registration.json`: immutable external-target identity, parser or serving contract, and compatibility metadata for a registered external model.
- `environment.json`: captured runtime, package, platform, and snapshot environment for that external target registration.
- `smoke_check.json`: the smallest sanity-check prompt or parser result for a registered external target; inspect this before trusting a full benchmark.
- `benchmark_summary.json`: top-line comparative benchmark summary, recommendation, and slice-level metrics.
- `scorecard_by_case.jsonl`: per-case comparative benchmark detail that explains surprising summary lines.
- `run_manifest.json`: per-target execution identity, config, serving identity, and lineage for one benchmark run.
- `raw_responses.jsonl`: exact captured model outputs for later debugging or parser inspection.
- `case_results.jsonl`: parsed and graded per-case benchmark results after the raw responses have been interpreted.
